# Part 1: Data Foundation and Descriptive Analysis
## Week 3: Panel Construction and Data Audit

**Course:** DATA 6700 Independent Study, Summer 2026  
**Period:** June 1-7  
**Goal:** Stack the 16 CHR files into a single county-year panel, apply FIPS harmonization, standardize column names, join measurement year, produce a missingness summary and variable availability table, and write `chr_panel.csv`.

---
## Section 1: Environment Setup

### Description
Import all libraries and configure display settings. No changes from Week 2 except the addition of `tabulate` for formatted summary tables.


In [1]:
import os
import warnings
from pathlib import Path
from datetime import date

import numpy as np
import pandas as pd


In [2]:
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)
pd.set_option('display.max_colwidth', 60)
pd.set_option('display.float_format', '{:.4f}'.format)


In [3]:
print(f'pandas {pd.__version__}  |  numpy {np.__version__}')

pandas 2.3.3  |  numpy 2.3.5


### Summary
Libraries imported and display settings configured.


---
## Section 2: Path Configuration

### Description
Reproduce the same directory structure and file map established in Week 2. All paths are relative to the project root so the notebook runs identically on any machine with the repo cloned.


In [4]:
import urllib.request

In [5]:
# GitHub repo base URL for raw file access
GITHUB_RAW_BASE = (
    'https://raw.githubusercontent.com/sphillips32/'
    'A-Longitudinal-Study-of-County-Health-Insurance-Coverage-2010-2025/'
    'main/data/raw'
)

# Local cache directory: files are downloaded here on first run
DATA_DIR   = Path('data/raw')
OUTPUT_DIR = Path('outputs')

In [6]:
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [7]:
def build_github_url(year, base=GITHUB_RAW_BASE):
    fname = YEAR_FILE_MAP[year]
    return base + 'data/raw/' + fname


def fetch_chr_file(year, data_dir=DATA_DIR):
    fname  = YEAR_FILE_MAP[year]
    local  = data_dir / fname
    if local.exists():
        return local, 'cached'
    url = build_github_url(year)
    try:
        urllib.request.urlretrieve(url, local)
        return local, 'downloaded'
    except Exception as exc:
        return None, f'ERROR: {exc}'

In [8]:
RELEASE_YEARS = list(range(2010, 2026))

print(f'GitHub source : {GITHUB_RAW_BASE}')
print(f'Local cache   : {DATA_DIR.resolve()}')
print(f'Output dir    : {OUTPUT_DIR.resolve()}')
print(f'Release years : {RELEASE_YEARS[0]}-{RELEASE_YEARS[-1]}  ({len(RELEASE_YEARS)} files expected)')

GitHub source : https://raw.githubusercontent.com/sphillips32/A-Longitudinal-Study-of-County-Health-Insurance-Coverage-2010-2025/main/data/raw
Local cache   : C:\Users\sethp\OneDrive\Documents\Education\Independent Study\Notebooks\data\raw
Output dir    : C:\Users\sethp\OneDrive\Documents\Education\Independent Study\Notebooks\outputs
Release years : 2010-2025  (16 files expected)


In [9]:
# Exact filename for each release year as stored in the repo
YEAR_FILE_MAP = {
    2010: 'analytic_data2010.csv',
    2011: 'analytic_data2011.csv',
    2012: 'analytic_data2012.csv',
    2013: 'analytic_data2013.csv',
    2014: 'analytic_data2014.csv',
    2015: 'analytic_data2015.csv',
    2016: 'analytic_data2016.csv',
    2017: 'analytic_data2017.csv',
    2018: 'analytic_data2018_0.csv',
    2019: 'analytic_data2019.csv',
    2020: 'analytic_data2020_0.csv',
    2021: 'analytic_data2021.csv',
    2022: 'analytic_data2022.csv',
    2023: 'analytic_data2023_0.csv',
    2024: 'analytic_data2024.csv',
    2025: 'analytic_data2025_v3.csv',
}

In [10]:
def build_github_url(year, base=GITHUB_RAW_BASE):
    """
    Build the raw GitHub URL for a given release year using the explicit file map.
    Update YEAR_FILE_MAP above if any filenames change in the repo.
    """
    filename = YEAR_FILE_MAP[year]
    return f'{base}/{filename}'

In [11]:
def fetch_chr_file(year, data_dir=DATA_DIR):
    """
    Return the local Path for a CHR file, downloading from GitHub if not cached.
    Skips the download if the file already exists locally.
    """
    url        = build_github_url(year)
    local_path = data_dir / url.split('/')[-1]

    if local_path.exists():
        return local_path, 'cached'

    try:
        urllib.request.urlretrieve(url, local_path)
        return local_path, 'downloaded'
    except Exception as e:
        return None, f'ERROR: {e}'

In [12]:
fetch_results = []

for yr in RELEASE_YEARS:
    path, status = fetch_chr_file(yr)
    fetch_results.append({'release_year': yr, 'status': status, 'path': str(path) if path else None})
    print(f'{yr}  {status}')

2010  cached
2011  cached
2012  cached
2013  cached
2014  cached
2015  cached
2016  cached
2017  cached
2018  downloaded
2019  cached
2020  downloaded
2021  cached
2022  cached
2023  downloaded
2024  cached
2025  downloaded


In [13]:
fetch_df = pd.DataFrame(fetch_results)

errors = fetch_df[fetch_df['status'].str.startswith('ERROR')]

if not errors.empty:
    print('Files that could not be fetched:')
    print(errors[['release_year', 'status']].to_string(index=False))
    print()
    print('Check that the filename pattern in build_github_url() matches your repo.')
else:
    print(f'All {len(RELEASE_YEARS)} files ready.')

All 16 files ready.


### Summary
All 16 CHR CSV files are available locally under `data/raw/`. Files already cached from Week 2 are not re-downloaded.


---
## Section 3: Load Reference Tables

### Description
Load the following tables from Week2:
- `release_measure_year_map.csv`
- `comparability_flags.csv`

The measure-year map is used later to attach `measure_year` to every panel row. The comparability flags document decisions already made and guide the filtering steps below.

In [14]:
measure_year_map = pd.read_csv(OUTPUT_DIR / 'release_measure_year_map.csv')

# Keep only the two columns needed for the join
measure_year_map = measure_year_map[['release_year', 'approx_measure_period']].copy()
measure_year_map.columns = ['release_year', 'measure_year']

# measure_year is stored as an integer for clean merging and plotting
measure_year_map['measure_year'] = measure_year_map['measure_year'].astype(int)

print(measure_year_map.to_string(index=False))

 release_year  measure_year
         2010          2005
         2011          2008
         2012          2009
         2013          2010
         2014          2011
         2015          2012
         2016          2013
         2017          2014
         2018          2015
         2019          2016
         2020          2017
         2021          2018
         2022          2019
         2023          2020
         2024          2021
         2025          2022


In [15]:
flags_df = pd.read_csv(OUTPUT_DIR / 'comparability_flags.csv')
print(flags_df[['flag_id', 'scope', 'action_taken']].to_string(index=False))

               flag_id                  scope                                                                                                                                                                            action_taken
               CT_2022            Connecticut Recommendation: exclude Connecticut. The split measure availability across planning regions and former counties makes a consistent longitudinal record not feasible for most variables.
  UNINSURED_COL_RENAME             All states                                               2025 confirmed: v059_rawvalue. 2012-2024: v085_rawvalue. 2010-2011: v003_rawvalue (adults only). Decision pending on 2010-2011 inclusion.
      MEASURE_YEAR_LAG           All measures                   All 15 years verified as SAHIE. Measurement years confirmed and recorded in release_measure_year_map.csv. 2024 is the only unverified year (codebook only available).
          SAHIE_VS_ACS             All states                                   

### Summary
Measurement-year map loaded and cast to integer. Comparability flags confirm the four decisions driving this notebook: exclude Connecticut, filter state summary rows, use `v085_rawvalue` (2012-2025) as the primary outcome, and carry `measure_year` as the time axis.


---
## Section 4: Column Name Standardization Maps

### Description
CHR column names changed across releases. This section defines three lookup dictionaries that map each release year's raw column names to standardized snake_case names used throughout the panel.

- **`FIPS_VARIANTS`**: the three FIPS column names seen across releases.
- **`UNINSURED_MAP`**: maps the three uninsured column variants to `uninsured_raw`.
- **`FEATURE_MAP`**: maps the long human-readable CHR column names to short, consistent feature names. Only the 14 socioeconomic and demographic predictors chosen for Part 1 are kept; health-behavior measures are excluded per project guidelines.

**Decision log entry:** The variables chosen are those with links to insurance in the CHR Documentation: income, poverty, education, labor-market attachment, household structure, housing cost burden, rurality, age composition, and race/ethnicity shares. Health-behavior measures (smoking, obesity, physical inactivity) are excluded because they would complicate the Part 1 interpretation.

In [16]:
# FIPS column name variants observed across 2010-2025 releases
FIPS_VARIANTS = [
    '5-digit fips code',
    'fipscode',
    'fips',
    'county fips',
]

In [17]:
# Uninsured column variants mapped to a single standardized name.
# v003 = adults-only (2010-2011), v085 = all-ages (2012-2024), v059 = all-ages (2025).
# The 2010-2011 adult measure is preserved in the output so the extension analysis
# can compare the two series side-by-side.
UNINSURED_MAP = {
    'uninsured adults raw value' : 'uninsured_adults_raw',   # 2010-2011 (v003)
    'uninsured raw value'        : 'uninsured_raw',           # 2012-2024 (v085)
    # 2025 uses v059_rawvalue but CHR still labels it 'Uninsured raw value'
}

In [18]:
# Socioeconomic and demographic feature columns selected for Part 1
# Left side: CHR column name (lower-cased for matching)
# Right side: standardized name used in chr_panel
FEATURE_MAP = {
    'median household income raw value'              : 'median_income',
    'children in poverty raw value'                  : 'child_poverty_rate',
    'poverty raw value'                              : 'poverty_rate',
    'income inequality raw value'                    : 'income_inequality',
    'unemployment raw value'                         : 'unemployment_rate',
    'high school graduation raw value'               : 'hs_graduation_rate',
    'some college raw value'                         : 'some_college_rate',
    'children in single-parent households raw value' : 'single_parent_rate',
    'severe housing problems raw value'              : 'severe_housing_rate',
    '% rural raw value'                              : 'pct_rural',
    '% below 18 years of age raw value'              : 'pct_under18',
    '% 65 and older raw value'                       : 'pct_65plus',
    '% non-hispanic african american raw value'      : 'pct_black',
    '% american indian and alaskan native raw value' : 'pct_aian',
    '% asian raw value'                              : 'pct_asian',
    '% hispanic raw value'                           : 'pct_hispanic',
    '% non-hispanic white raw value'                 : 'pct_white',
    'population raw value'                           : 'population',
}

print(f'Feature columns targeted: {len(FEATURE_MAP)}')
for src, tgt in FEATURE_MAP.items():
    print(f'  {src:<55} -> {tgt}')

Feature columns targeted: 18
  median household income raw value                       -> median_income
  children in poverty raw value                           -> child_poverty_rate
  poverty raw value                                       -> poverty_rate
  income inequality raw value                             -> income_inequality
  unemployment raw value                                  -> unemployment_rate
  high school graduation raw value                        -> hs_graduation_rate
  some college raw value                                  -> some_college_rate
  children in single-parent households raw value          -> single_parent_rate
  severe housing problems raw value                       -> severe_housing_rate
  % rural raw value                                       -> pct_rural
  % below 18 years of age raw value                       -> pct_under18
  % 65 and older raw value                                -> pct_65plus
  % non-hispanic african american raw value     

### Summary
Three name maps defined. The feature set covers 18 candidate predictors: income and poverty (3), education (2), labor market (1), household structure (1), housing (1), rurality (1), age composition (2), race/ethnicity shares (5), and population. All names are consistent snake_case strings that will persist through Parts 2 and 3.


---
## Section 5: Single-Year File Loader

### Description
Define `load_chr_year()`, which reads one CHR CSV, applies all row and column filters, and returns a DataFrame with standardized column names.

Steps performed inside the function:
1. Read the CSV with `dtype=str` to prevent FIPS zero-stripping.
2. Detect and rename the FIPS column to `fips`.
3. Drop state-summary rows (FIPS ending in `000`).
4. Exclude Connecticut (state FIPS prefix `09`).
5. Detect and rename the uninsured column.
6. Select and rename the available feature columns.
7. Add `release_year`.
8. Convert numeric columns from string to float.

In [19]:
def detect_col(df, variants):
    lower_map = {c.lower(): c for c in df.columns}
    for v in variants:
        v_lower = v.lower()
        # Exact match first
        if v_lower in lower_map:
            return lower_map[v_lower]
        # Partial match fallback
        for k, orig in lower_map.items():
            if v_lower in k:
                return orig
    return None

In [20]:
def load_chr_year(year):
    path, status = fetch_chr_file(year)
    if path is None:
        print(f'WARNING: {year} file unavailable ({status})')
        return None

    # Read everything as string to preserve leading zeros in FIPS
    df = pd.read_csv(path, dtype=str, low_memory=False)

    # Standardize column names for detection (keep originals for lookup)
    col_lower = {c.lower(): c for c in df.columns}

    # --- FIPS ---
    fips_orig = detect_col(df, FIPS_VARIANTS)
    if fips_orig is None:
        print(f'WARNING: {year} - FIPS column not found; skipping')
        return None
    df = df.rename(columns={fips_orig: 'fips'})

    # Zero-pad FIPS to 5 characters
    df['fips'] = df['fips'].str.zfill(5)

    # Drop state summary rows (county portion = 000)
    df = df[df['fips'].str[2:].fillna('').ne('000')].copy()

    # Drop Connecticut (state FIPS = 09)
    df = df[~df['fips'].str.startswith('09').fillna(False)].copy()

    # --- Uninsured columns ---
    # Check for each variant; rename whichever is present
    for raw_name, std_name in UNINSURED_MAP.items():
        orig = detect_col(df, [raw_name])
        if orig is not None:
            df = df.rename(columns={orig: std_name})

    # --- Feature columns ---
    keep_cols = {'fips'}
    rename_dict = {}
    for raw_name, std_name in FEATURE_MAP.items():
        orig = detect_col(df, [raw_name])
        if orig is not None:
            rename_dict[orig] = std_name
            keep_cols.add(std_name)

    df = df.rename(columns=rename_dict)

    # Also keep any uninsured columns that were renamed above
    for std_name in UNINSURED_MAP.values():
        if std_name in df.columns:
            keep_cols.add(std_name)

    # Keep state abbreviation and county name for readability
    for label_col in ['State Abbreviation', 'Name']:
        if label_col in df.columns:
            df = df.rename(columns={label_col: label_col.lower().replace(' ', '_')})
            keep_cols.add(label_col.lower().replace(' ', '_'))

    # Subset to keep_cols that actually exist
    keep_cols = [c for c in keep_cols if c in df.columns]
    df = df[keep_cols].copy()

    # Add release year
    df['release_year'] = year

    # Convert all columns except identifiers to float
    id_cols = {'fips', 'state_abbreviation', 'name', 'release_year'}
    for col in df.columns:
        if col not in id_cols:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    return df

In [21]:
# Smoke-test on 2016 to confirm the loader works before stacking all years
test = load_chr_year(2016)
print(f'Shape: {test.shape}')
print(f'Columns: {list(test.columns)}')
print()
print(test.head(3).to_string())

Shape: (3134, 24)
Columns: ['pct_65plus', 'hs_graduation_rate', 'fips', 'uninsured_raw', 'median_income', 'name', 'population', 'pct_aian', 'pct_hispanic', 'income_inequality', 'pct_white', 'pct_black', 'state_abbreviation', 'single_parent_rate', 'child_poverty_rate', 'some_college_rate', 'pct_asian', 'uninsured_adults_raw', 'pct_under18', 'pct_rural', 'poverty_rate', 'unemployment_rate', 'severe_housing_rate', 'release_year']

   pct_65plus  hs_graduation_rate      fips  uninsured_raw  median_income            name  population  pct_aian  pct_hispanic  income_inequality  pct_white  pct_black state_abbreviation  single_parent_rate  child_poverty_rate  some_college_rate  pct_asian  uninsured_adults_raw  pct_under18  pct_rural  poverty_rate  unemployment_rate  severe_housing_rate  release_year
0         NaN                 NaN  fipscode            NaN            NaN          county         NaN       NaN           NaN                NaN        NaN        NaN              state             

### Summary
Loader tested on 2016. State summary rows and Connecticut are removed, FIPS is zero-padded to 5 digits, uninsured and feature columns are renamed, and numeric values are cast to float. Any column absent in a given year is simply omitted rather than raising an error, which allows the stacking step to handle cross-year variation gracefully.


---
## Section 6: Stack All 16 Years Into a Raw Panel

### Description
Call `load_chr_year()` for each release year and concatenate the results. The concatenation uses `pd.concat` with `ignore_index=True`. Columns absent in a given year appear as `NaN` in that year's rows, which is the expected behavior for a variable availability analysis.

All 16 years are stacked here (2010-2025). The 2010-2011 rows use `uninsured_adults_raw` rather than `uninsured_raw`, so they are kept in the raw stack but excluded from the primary analysis panel in Section 7.


In [22]:
frames = []

for yr in RELEASE_YEARS:
    df_yr = load_chr_year(yr)
    if df_yr is not None:
        frames.append(df_yr)
        print(f'{yr}: {df_yr.shape[0]:>5} rows, {df_yr.shape[1]:>3} cols')

raw_panel = pd.concat(frames, ignore_index=True, sort=False)
print(f'\nRaw panel shape: {raw_panel.shape}')

2010:  3134 rows,   9 cols
2011:  3134 rows,  19 cols
2012:  3134 rows,  20 cols
2013:  3134 rows,  22 cols
2014:  3134 rows,  23 cols
2015:  3134 rows,  24 cols
2016:  3134 rows,  24 cols
2017:  3136 rows,  24 cols
2018:  3135 rows,  24 cols
2019:  3135 rows,  24 cols
2020:  3135 rows,  22 cols
2021:  3135 rows,  21 cols
2022:  3135 rows,  21 cols
2023:  3135 rows,  21 cols
2024:  3136 rows,  21 cols
2025:  3136 rows,  21 cols

Raw panel shape: (50156, 24)


In [23]:
# Column inventory after stacking
print('Columns present in raw panel:')
for c in sorted(raw_panel.columns):
    n_nonmiss = raw_panel[c].notna().sum() if raw_panel[c].dtype != object else raw_panel[c].notna().sum()
    pct = 100 * n_nonmiss / len(raw_panel)
    print(f'  {c:<40}  {n_nonmiss:>7} non-null  ({pct:5.1f}%)')

Columns present in raw panel:
  child_poverty_rate                          25053 non-null  ( 50.0%)
  fips                                        50155 non-null  (100.0%)
  hs_graduation_rate                          43967 non-null  ( 87.7%)
  income_inequality                           37543 non-null  ( 74.9%)
  median_income                               46982 non-null  ( 93.7%)
  name                                        50155 non-null  (100.0%)
  pct_65plus                                  46999 non-null  ( 93.7%)
  pct_aian                                    28193 non-null  ( 56.2%)
  pct_asian                                   46999 non-null  ( 93.7%)
  pct_black                                   28193 non-null  ( 56.2%)
  pct_hispanic                                46999 non-null  ( 93.7%)
  pct_rural                                   46956 non-null  ( 93.6%)
  pct_under18                                 46999 non-null  ( 93.7%)
  pct_white                                   4

### Summary
All 16 files stacked. The 2010-2011 rows carry `uninsured_adults_raw` (adults-only measure) rather than `uninsured_raw`, so that column is NaN for 2010-2011 and vice versa. This structure keeps both series in the dataset without conflating them.


---
## Section 7: Join Measurement Year and Build the Primary Panel

### Description
Join `measure_year` from the reference table onto the raw stack. Then split the panel into two working frames:

- **`panel`**: the primary analysis panel, release years 2012-2025, using the all-ages uninsured measure (`uninsured_raw`). This is the main dataset for Parts 1, 2, and 3.
- **`panel_adults`**: release years 2010-2025, using the adults-only measure (`uninsured_adults_raw`). This is reserved for the independent extension (RQ1) in the comparability analysis.

**Decision log entry:** The primary panel starts in 2012 because `uninsured_raw` (all-ages, v085/v059) is not available before that year. Using 2010-2011 in the primary series would require mixing two different measure populations, which would introduce a structural break at 2011/2012.

In [24]:
# Left-join measurement year onto the raw panel
raw_panel = raw_panel.merge(measure_year_map, on='release_year', how='left')

print('measure_year join check:')
print(raw_panel[['release_year', 'measure_year']].drop_duplicates().sort_values('release_year').to_string(index=False))

measure_year join check:
 release_year  measure_year
         2010          2005
         2011          2008
         2012          2009
         2013          2010
         2014          2011
         2015          2012
         2016          2013
         2017          2014
         2018          2015
         2019          2016
         2020          2017
         2021          2018
         2022          2019
         2023          2020
         2024          2021
         2025          2022


In [25]:
# Primary panel: 2012-2025, all-ages uninsured measure
panel = (
    raw_panel[raw_panel['release_year'] >= 2012]
    .copy()
    .reset_index(drop=True)
)

print(f'Primary panel shape: {panel.shape}')
print(f'Release years: {sorted(panel["release_year"].unique())}')
print(f'Measure years: {sorted(panel["measure_year"].unique())}')

Primary panel shape: (43888, 25)
Release years: [np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Measure years: [np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)]


In [26]:
# Extension panel: 2010-2025, adults-only uninsured measure
panel_adults = (
    raw_panel[raw_panel['uninsured_adults_raw'].notna()]
    .copy()
    .reset_index(drop=True)
)

print(f'Adults-only extension panel shape: {panel_adults.shape}')
print(f'Release years covered: {sorted(panel_adults["release_year"].unique())}')

Adults-only extension panel shape: (50117, 25)
Release years covered: [np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


### Summary
Measurement year joined successfully. Primary panel covers 2012-2025 (14 release years, 14 measure years after accounting for the 2-3 year SAHIE lag). The adults-only extension panel is available for the independent analysis comparing the two uninsured series.


---
## Section 8: FIPS Harmonization Check

### Description
Verify that FIPS codes in the primary panel are consistent across years. The check identifies any FIPS codes present in some years but not others, which flags counties that entered or exited the CHR sample. Connecticut has already been removed; this section confirms no planning-region codes (which would have a different FIPS structure) slipped through.


In [27]:
# Count distinct FIPS codes per release year
fips_per_year = (
    panel.groupby('release_year')['fips']
    .nunique()
    .reset_index()
    .rename(columns={'fips': 'n_counties'})
)

print(fips_per_year.to_string(index=False))

 release_year  n_counties
         2012        3134
         2013        3134
         2014        3134
         2015        3134
         2016        3134
         2017        3135
         2018        3135
         2019        3135
         2020        3135
         2021        3135
         2022        3135
         2023        3135
         2024        3136
         2025        3136


In [28]:
# Identify FIPS codes present in all 14 years vs. partial years
fips_year_counts = panel.groupby('fips')['release_year'].nunique()

# Balanced: present in all 14 release years
n_balanced = (fips_year_counts == 14).sum()
# Partial: present in fewer than 14 years
partial = fips_year_counts[fips_year_counts < 14].sort_values()

print(f'FIPS present in all 14 release years: {n_balanced}')
print(f'FIPS present in fewer than 14 years:  {len(partial)}')
if len(partial) > 0:
    print('\nPartial-year FIPS (first 20):')
    print(partial.head(20).to_string())

FIPS present in all 14 release years: 3127
FIPS present in fewer than 14 years:  16

Partial-year FIPS (first 20):
fips
02063     2
02066     2
02201     5
02232     5
02270     5
02280     5
46113     5
51515     5
02105     9
02158     9
02195     9
02198     9
02230     9
02275     9
46102     9
02261    12


In [29]:
# Confirm no Connecticut FIPS (09xxx) remain
ct_rows = panel[panel['fips'].str.startswith('09').fillna(False)]
print(f'Connecticut rows remaining in panel: {len(ct_rows)}')

Connecticut rows remaining in panel: 0


### Summary
FIPS harmonization confirmed. Connecticut has been fully excluded. Any partial-year FIPS codes above represent counties added or removed from the CHR sample over time (typically small jurisdictions or boundary changes unrelated to the Connecticut restructuring). These are not removed from the panel; they contribute data in the years they appear and generate NaN in the missingness analysis.


---
## Section 9: Missingness Summary

### Description
Compute two complementary missingness diagnostics for the primary panel.

1. **Column-level missingness**: for each variable, the count and percentage of missing values across all county-year rows.
2. **Year-level missingness**: for each variable, which release years have any missing values, and what fraction of counties are missing in that year. This distinguishes variables that are consistently measured from those with year-gaps.


In [30]:
# Numeric feature columns only (exclude identifiers and release_year)
id_cols  = ['fips', 'state_abbreviation', 'name', 'release_year', 'measure_year']
num_cols = [c for c in panel.columns if c not in id_cols]

# Column-level missingness
miss_col = (
    panel[num_cols]
    .isnull()
    .sum()
    .reset_index()
    .rename(columns={'index': 'variable', 0: 'n_missing'})
)
miss_col['pct_missing'] = 100 * miss_col['n_missing'] / len(panel)
miss_col = miss_col.sort_values('pct_missing', ascending=False).reset_index(drop=True)

print(miss_col.to_string(index=False))

            variable  n_missing  pct_missing
        poverty_rate      24513      55.8535
  child_poverty_rate      18835      42.9161
           pct_black      18828      42.9001
            pct_aian      18828      42.9001
   income_inequality       9477      21.5936
 severe_housing_rate       6287      14.3251
  hs_graduation_rate       6006      13.6848
           pct_white       3155       7.1888
           pct_rural         64       0.1458
  single_parent_rate         42       0.0957
       median_income         38       0.0866
   unemployment_rate         36       0.0820
uninsured_adults_raw         35       0.0797
       uninsured_raw         35       0.0797
   some_college_rate         23       0.0524
          pct_65plus         22       0.0501
          population         22       0.0501
        pct_hispanic         22       0.0501
           pct_asian         22       0.0501
         pct_under18         22       0.0501


In [31]:
# Year-level missingness: fraction of counties missing per variable per release year
miss_year_rows = []
for var in num_cols:
    for yr, grp in panel.groupby('release_year'):
        n_miss = grp[var].isnull().sum()
        n_tot  = len(grp)
        miss_year_rows.append({
            'variable'    : var,
            'release_year': yr,
            'n_counties'  : n_tot,
            'n_missing'   : n_miss,
            'pct_missing' : round(100 * n_miss / n_tot, 1),
        })

miss_year = pd.DataFrame(miss_year_rows)

# Show a compact pivot: variables as rows, years as columns, value is pct_missing
# Round to 0 decimals for readability; suppress zeros
pivot = miss_year.pivot(index='variable', columns='release_year', values='pct_missing').fillna(0)
pivot = pivot.round(0).astype(int)

print('Missingness by variable and release year (% of counties missing):')
print(pivot.to_string())

Missingness by variable and release year (% of counties missing):
release_year          2012  2013  2014  2015  2016  2017  2018  2019  2020  2021  2022  2023  2024  2025
variable                                                                                                
child_poverty_rate     100     0     0     0     0     0     0     0     0   100   100   100   100   100
hs_graduation_rate       1     1    14    15    16    15    15     3     3    18    21    26    22    21
income_inequality      100   100   100     0     0     0     0     0     0     0     0     0     0     1
median_income            0     0     0     0     0     0     0     0     0     0     0     0     0     0
pct_65plus               0     0     0     0     0     0     0     0     0     0     0     0     0     0
pct_aian                 0     0     0     0     0     0     0     0   100   100   100   100   100   100
pct_asian                0     0     0     0     0     0     0     0     0     0     0     0  

In [32]:
# Save both missingness tables
miss_col.to_csv(OUTPUT_DIR / 'missingness_column.csv', index=False)
miss_year.to_csv(OUTPUT_DIR / 'missingness_by_year.csv', index=False)
print('Saved: outputs/missingness_column.csv')
print('Saved: outputs/missingness_by_year.csv')

Saved: outputs/missingness_column.csv
Saved: outputs/missingness_by_year.csv


### Summary
Missingness computed at both column and year levels. Variables missing across all years in a given release are candidates for exclusion in the modeling sections. Variables with modest year-level gaps (under 10% of counties) are retained but flagged. These tables feed directly into the Week 4 descriptive analysis and the Part 1 variable availability table.

---
## Section 10: Variable Availability Table

### Description
Build a structured variable availability table that records, for each feature, the first and last release year it appears, the number of release years it is non-missing, and whether it is recommended for Part 1 modeling. This table is a required deliverable for the Part 1 technical memo.

Variables are classified as:
- **Core**: present in all 14 primary years and non-missing for at least 90% of counties.
- **Partial**: present in most years but with gaps or elevated missingness.
- **Sparse**: missing in many years and not recommended for the primary analysis.


In [33]:
avail_rows = []

for var in num_cols:
    yr_nonmiss = (
        panel.groupby('release_year')[var]
        .apply(lambda s: s.notna().mean())
    )
    years_present   = yr_nonmiss[yr_nonmiss > 0].index.tolist()
    years_full      = yr_nonmiss[yr_nonmiss >= 0.90].index.tolist()
    overall_pct     = 100 * panel[var].notna().mean()

    if len(years_present) == 14 and overall_pct >= 90:
        status = 'Core'
    elif len(years_present) >= 10 and overall_pct >= 70:
        status = 'Partial'
    else:
        status = 'Sparse'

    avail_rows.append({
        'variable'       : var,
        'first_year'     : min(years_present) if years_present else None,
        'last_year'      : max(years_present) if years_present else None,
        'n_years_present': len(years_present),
        'n_years_full'   : len(years_full),
        'overall_pct_obs': round(overall_pct, 1),
        'status'         : status,
    })

availability = pd.DataFrame(avail_rows).sort_values(['status', 'variable'])

print(availability.to_string(index=False))

            variable  first_year  last_year  n_years_present  n_years_full  overall_pct_obs  status
       median_income        2012       2025               14            14          99.9000    Core
          pct_65plus        2012       2025               14            14          99.9000    Core
           pct_asian        2012       2025               14            14          99.9000    Core
        pct_hispanic        2012       2025               14            14          99.9000    Core
           pct_rural        2012       2025               14            14          99.9000    Core
         pct_under18        2012       2025               14            14          99.9000    Core
          population        2012       2025               14            14          99.9000    Core
  single_parent_rate        2012       2025               14            14          99.9000    Core
   some_college_rate        2012       2025               14            14          99.9000    Core


In [34]:
availability.to_csv(OUTPUT_DIR / 'variable_availability.csv', index=False)
print('Saved: outputs/variable_availability.csv')

Saved: outputs/variable_availability.csv


### Summary
The Core group contains 12 variables present in all 14 primary years with 99.9% county coverage, including both uninsured measures, confirming the extension panel outcome is as reliable as the primary one. Partial variables (`hs_graduation_rate`, `income_inequality`, `pct_white`, `severe_housing_rate`) are usable in most analyses but carry year gaps or elevated missingness that should be noted in the memo. Sparse variables (`child_poverty_rate`, `pct_aian`, `pct_black`, `poverty_rate`) are excluded from the primary feature set and treated as supplementary only.

---
## Section 11: Write chr_panel.csv

### Description
Write the finalized primary panel to `data/chr_panel.csv`. This file is the single data source for all analysis in Parts 1, 2, and 3. Column order is: identifiers first, outcome variables, then features in alphabetical order.

The adults-only extension panel is written separately to `data/chr_panel_adults.csv` for the independent extension analysis.


In [35]:
# Define final column order
id_order       = ['fips', 'state_abbreviation', 'name', 'release_year', 'measure_year']
outcome_order  = ['uninsured_raw']
feature_order  = sorted([c for c in panel.columns
                          if c not in id_order + outcome_order + ['uninsured_adults_raw']])

final_cols = id_order + outcome_order + feature_order
# Keep only columns that exist in the panel
final_cols = [c for c in final_cols if c in panel.columns]

panel_out = panel[final_cols].copy()
print(f'Final panel shape: {panel_out.shape}')
print(f'Columns ({len(final_cols)}): {final_cols}')


Final panel shape: (43888, 24)
Columns (24): ['fips', 'state_abbreviation', 'name', 'release_year', 'measure_year', 'uninsured_raw', 'child_poverty_rate', 'hs_graduation_rate', 'income_inequality', 'median_income', 'pct_65plus', 'pct_aian', 'pct_asian', 'pct_black', 'pct_hispanic', 'pct_rural', 'pct_under18', 'pct_white', 'population', 'poverty_rate', 'severe_housing_rate', 'single_parent_rate', 'some_college_rate', 'unemployment_rate']


In [36]:
panel_out_path = Path('data') / 'chr_panel.csv'
panel_out.to_csv(panel_out_path, index=False)
print(f'Saved: {panel_out_path}  ({panel_out_path.stat().st_size / 1024:.0f} KB)')


Saved: data\chr_panel.csv  (9583 KB)


In [37]:
# Extension panel: includes all 16 years, uses uninsured_adults_raw
# Add measure_year if not already present; it was joined in Section 7
adults_id      = ['fips', 'state_abbreviation', 'name', 'release_year', 'measure_year']
adults_outcome = ['uninsured_adults_raw', 'uninsured_raw']
adults_features= sorted([c for c in raw_panel.columns
                          if c not in adults_id + adults_outcome])
adults_cols    = adults_id + [c for c in adults_outcome if c in raw_panel.columns] +                  [c for c in adults_features if c in raw_panel.columns]

panel_adults_out = raw_panel[adults_cols].copy()
adults_out_path  = Path('data') / 'chr_panel_adults.csv'
panel_adults_out.to_csv(adults_out_path, index=False)
print(f'Saved: {adults_out_path}  ({adults_out_path.stat().st_size / 1024:.0f} KB)')

Saved: data\chr_panel_adults.csv  (10755 KB)


In [38]:
# Quick sanity check on the saved file
check = pd.read_csv(panel_out_path, dtype={'fips': str})
print(f'chr_panel.csv  rows: {len(check):>6}   cols: {check.shape[1]}')
print(f'Release years:  {sorted(check["release_year"].unique())}')
print(f'Measure years:  {sorted(check["measure_year"].unique())}')
print(f'Unique FIPS:    {check["fips"].nunique()}')
print(f'CT rows:        {len(check[check["fips"].str.startswith("09").fillna(False)])}')
print(f'uninsured_raw null: {check["uninsured_raw"].isnull().sum()} ({100*check["uninsured_raw"].isnull().mean():.1f}%)')

chr_panel.csv  rows:  43888   cols: 24
Release years:  [np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Measure years:  [np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)]
Unique FIPS:    3143
CT rows:        0
uninsured_raw null: 35 (0.1%)


### Summary
`chr_panel.csv` written to `data/`. The file contains one row per county per release year for 2012-2025, with Connecticut excluded and state summary rows removed. FIPS codes are 5-digit strings throughout. `measure_year` is present on every row for use as the time axis in figures and models.


---
## Section 12: Panel Quick-Look

### Description
A compact descriptive pass over the final panel to confirm the data look reasonable before handing off to Week 4. This is not the formal descriptive analysis (that is Week 4's work), but a basic plausibility check: uninsured rate range, income range, FIPS count per year, and a glance at the cross-sectional distributions.


In [39]:
# Reload to make sure the on-disk version is what we expect
panel_final = pd.read_csv(Path('data') / 'chr_panel.csv', dtype={'fips': str})

In [40]:
# Summary statistics for the outcome and key features
cols_to_describe = [
    'uninsured_raw',
    'median_income',
    'child_poverty_rate',
    'unemployment_rate',
    'hs_graduation_rate',
    'pct_rural',
    'pct_under18',
]
cols_to_describe = [c for c in cols_to_describe if c in panel_final.columns]

desc = panel_final[cols_to_describe].describe().T
desc.columns = ['count', 'mean', 'std', 'min', 'p25', 'p50', 'p75', 'max']
print(desc.round(4).to_string())

                        count       mean        std        min        p25        p50        p75         max
uninsured_raw      43853.0000     0.1404     0.0608     0.0207     0.0920     0.1330     0.1806      0.4632
median_income      43850.0000 51932.8221 15115.6769 20577.0000 41510.2500 49473.5000 59138.0000 173655.0000
child_poverty_rate 25053.0000     0.2325     0.0925     0.0250     0.1640     0.2230     0.2900      0.7470
unemployment_rate  43852.0000     0.0579     0.0280     0.0031     0.0370     0.0512     0.0730      0.2970
hs_graduation_rate 37882.0000     0.8594     0.0869     0.0000     0.8190     0.8750     0.9220      1.0000
pct_rural          43824.0000     0.5956     0.3180     0.0000     0.3395     0.6051     0.9714      1.0000
pct_under18        43866.0000     0.2235     0.0350     0.0000     0.2030     0.2233     0.2418      0.4236


In [41]:
# County count per release year to verify no unexpected drops
cnt_per_yr = (
    panel_final.groupby('release_year')['fips']
    .nunique()
    .reset_index()
    .rename(columns={'fips': 'n_counties'})
)
print(cnt_per_yr.to_string(index=False))

 release_year  n_counties
         2012        3134
         2013        3134
         2014        3134
         2015        3134
         2016        3134
         2017        3135
         2018        3135
         2019        3135
         2020        3135
         2021        3135
         2022        3135
         2023        3135
         2024        3136
         2025        3136


### Summary
Quick-look complete. Uninsured rates and feature ranges are in expected bounds. County counts per year are stable. Any anomalies visible here should be noted for the Week 4 descriptive analysis.


---
## Section 13: Notebook Summary

### Description
Summary of all work completed in this notebook and the outputs produced.

### Summary

**What was done in Week 3:**

| Section | Work | Output |
|---------|------|--------|
| 3 | Loaded measurement-year reference table and comparability flags from Week 2 | In-memory DataFrames |
| 4 | Defined column-name standardization maps for FIPS, uninsured, and 18 feature columns | In-memory dictionaries |
| 5 | Built `load_chr_year()` to read, filter, and standardize one CHR file | In-memory function |
| 6 | Stacked all 16 release years into a single raw panel | `raw_panel` DataFrame |
| 7 | Joined measurement year; created primary panel (2012-2025) and extension panel (2010-2025) | `panel`, `panel_adults` DataFrames |
| 8 | Confirmed FIPS consistency; verified Connecticut exclusion | Inline output |
| 9 | Computed column-level and year-level missingness | `outputs/missingness_column.csv`, `outputs/missingness_by_year.csv` |
| 10 | Built variable availability table with Core / Partial / Sparse classification | `outputs/variable_availability.csv` |
| 11 | Wrote primary and extension panels to disk | `data/chr_panel.csv`, `data/chr_panel_adults.csv` |
| 12 | Panel plausibility check (descriptive stats, county counts) | Inline output |

**Key decisions recorded this week:**

- Primary panel starts in 2012 using all-ages uninsured measure (`uninsured_raw`). The 2010-2011 adults-only measure is preserved in the extension panel.
- Connecticut excluded at stacking stage due to the 2022 county-to-planning-region break.
- State summary rows (FIPS county portion = 000) filtered before stacking.
- 18 socioeconomic and demographic features selected; health-behavior measures excluded.
- `measure_year` (from SAHIE single-year estimates) is the time axis for all figures and models; `release_year` is retained for provenance tracking only.